In [1]:
import json
import ast
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

def visualize_result(result, idx, save_dir="outputs"):
    os.makedirs(save_dir, exist_ok=True)

    # Load input image
    img_path = result["image_path"]
    img = mpimg.imread(img_path)

    ground_truth = ""#result["ground_truth"]
    prediction = result["model_output"]

    # Decide how many ranks to show (top 3)
    n_ranks = min(3, len(result["top_concepts_over_sequence"]))
    # Max number of crops across those ranks
    max_crops = max(len(ast.literal_eval(c["image_grounding_path"])) 
                    if isinstance(c["image_grounding_path"], str) else len(c["image_grounding_path"]) 
                    for c in result["top_concepts_over_sequence"][:n_ranks])

    # Layout: [big input image] + [similarity bar + crops...]
    fig = plt.figure(figsize=(4 + (max_crops+1)*2, n_ranks*2.8))
    gs = fig.add_gridspec(
        n_ranks, max_crops+2, 
        width_ratios=[2] + [0.3] + [1]*max_crops,
        wspace=0.05, hspace=0.05  # reduce spacing between cells
    )

    # Left side: input image (spans all rows)
    ax_left = fig.add_subplot(gs[:, 0])
    ax_left.imshow(img)
    ax_left.axis("off")
    match = "✅" if prediction.strip().lower() == ground_truth.strip().lower() else "❌"
    ax_left.set_title(f"GT: {ground_truth} | Pred: {prediction} {match}", fontsize=24)

    # Right side: rows for top ranks
    for rank in range(n_ranks):
        concept = result["top_concepts_over_sequence"][rank]
        paths_str = concept["image_grounding_path"]
        paths = ast.literal_eval(paths_str) if isinstance(paths_str, str) else paths_str
        similarity = concept.get("similarity", 0.0)

        # Add vertical similarity bar at column 1 (thin + orange)
        ax_bar = fig.add_subplot(gs[rank, 1])
        ax_bar.bar([0], [similarity], color="orange", width=0.2)
        ax_bar.set_ylim(0, 1)
        ax_bar.set_xticks([])
        ax_bar.set_yticks([0, 0.5, 1.0])
        ax_bar.tick_params(axis="y", labelsize=14)
        ax_bar.set_ylabel(f"{similarity:.2f}", fontsize=18, rotation=0, labelpad=15)
        ax_bar.set_title(f"Rank {rank+1}", fontsize=20, pad=8)

        # Add concept crops to the right
        for j, item in enumerate(paths):
            try:
                _, crop_path = item.split("@")
            except ValueError:
                crop_path = item

            ax = fig.add_subplot(gs[rank, j+2])
            if os.path.exists(crop_path):
                crop_img = mpimg.imread(crop_path)
                ax.imshow(crop_img)
            ax.axis("off")

        # Row label (text_grounding)
        text_label = ", ".join(concept.get("text_grounding", []))
        fig.text(0.35, 1 - (rank+1)/(n_ranks+0.2), text_label,
                 ha="left", va="center", fontsize=22, color="darkblue")

    # Make figure compact
    plt.subplots_adjust(wspace=0.05, hspace=0.05)
    save_path = os.path.join(save_dir, f"viz_{idx}.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved visualization: {save_path}")





def visualize_all(json_path, save_dir="outputs"):
    """
    Loop through all results in the JSON and visualize them.
    """
    with open(json_path, "r") as f:
        data = json.load(f)

    results = data["results"]
    for idx, result in enumerate(results):
        visualize_result(result, idx, save_dir)


# Example usage:
# visualize_all("your_file.json", save_dir="viz_outputs")


In [2]:
visualize_all("/mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_dl_imagenet_unsupervised/explanations/snmf/vlm_explanations.json", save_dir="/mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_dl_imagenet_unsupervised/plots")

/tmp/ipykernel_1209178/2682791391.py:77: UserWarning: Glyph 10060 (\N{CROSS MARK}) missing from font(s) DejaVu Sans.
  plt.savefig(save_path, dpi=150, bbox_inches="tight")


Saved visualization: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_dl_imagenet_unsupervised/plots/viz_0.png
Saved visualization: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_dl_imagenet_unsupervised/plots/viz_1.png
Saved visualization: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_dl_imagenet_unsupervised/plots/viz_2.png
Saved visualization: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_dl_imagenet_unsupervised/plots/viz_3.png
Saved visualization: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_dl_imagenet_unsupervised/plots/viz_4.png
Saved visualization: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_dl_imagenet_unsupervised/plots/viz_5.png
Saved visualization: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_dl_imagenet_unsupervised/plots/viz_6.png
Saved visualization: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_dl_imagenet_unsupervised/plots/viz_7.png
Saved visualization: /mnt/abka03

In [52]:
import torch

# Replace with your checkpoint path
path = "/mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_cgdl_imagenet_unsupervised/concept/snmf/combined_concept_snmf_raw.pth"

# Load checkpoint
checkpoint = torch.load(path, map_location="cpu")

# Show top-level keys
print("Keys in checkpoint:", checkpoint.keys())

Keys in checkpoint: dict_keys(['concepts', 'activations', 'decomposition_method', 'text_grounding', 'image_grounding_paths', 'analysis_model'])


In [53]:
import os
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def save_grounding_rows(image_grounding_paths, text_groundings, save_dir="groundings", max_rows=None):
    os.makedirs(save_dir, exist_ok=True)

    num_rows = len(image_grounding_paths)
    if max_rows:
        num_rows = min(num_rows, max_rows)

    for i in range(num_rows):
        tokens = text_groundings[i]
        paths = image_grounding_paths[i]

        # split path into prefix and suffix
        prefixes = [p.split("@")[0] for p in paths]
        suffixes = [p.split("@")[-1] for p in paths]

        # make a safe filename from unique prefixes
        unique_prefixes = sorted(set(prefixes))
        name_str = "_".join(unique_prefixes)
        name_str = name_str.replace("/", "_").replace(" ", "_")  # sanitize filename
        if len(name_str) > 100:  # avoid overly long filenames
            name_str = name_str[:100]

        # add random number to avoid overwriting
        rand_suffix = random.randint(1000, 9999)
        filename = f"{name_str}_{rand_suffix}.pdf"

        # figure size proportional to number of images (compact)
        fig, axes = plt.subplots(
            1, len(suffixes) + 1,
            figsize=(3*(len(suffixes)+1), 3),
            constrained_layout=True
        )

        # if only one row of axes, make iterable
        if len(suffixes) + 1 == 1:
            axes = [axes]

        # column 0: tokens as text
        axes[0].axis("off")
        axes[0].text(0.0, 0.5, "\n".join(tokens), fontsize=40,
                     va="center", ha="left", wrap=True)

        # columns 1..: images
        for j, ax in enumerate(axes[1:]):
            ax.axis("off")
            if os.path.exists(suffixes[j]):
                img = mpimg.imread(suffixes[j])
                ax.imshow(img)
            else:
                ax.text(0.5, 0.5, f"Missing:\n{suffixes[j]}",
                        ha="center", va="center", fontsize=10)

        # save without extra whitespace
        out_path = os.path.join(save_dir, filename)
        plt.savefig(out_path, dpi=150, bbox_inches="tight", pad_inches=0.1)
        plt.close(fig)
        print(f"Saved: {out_path}")


In [54]:
image_grounding_paths = checkpoint["image_grounding_paths"]
text_groundings = checkpoint["text_grounding"]


save_grounding_rows(image_grounding_paths, text_groundings, save_dir="/mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_cgdl_imagenet_unsupervised/plots/concept_image",max_rows=200)

Saved: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_cgdl_imagenet_unsupervised/plots/concept_image/dirt_8414.pdf
Saved: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_cgdl_imagenet_unsupervised/plots/concept_image/Green_8472.pdf
Saved: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_cgdl_imagenet_unsupervised/plots/concept_image/Nature_5099.pdf
Saved: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_cgdl_imagenet_unsupervised/plots/concept_image/Leash_7511.pdf
Saved: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_cgdl_imagenet_unsupervised/plots/concept_image/mammal_7584.pdf
Saved: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_cgdl_imagenet_unsupervised/plots/concept_image/Tooth_3098.pdf
Saved: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_cgdl_imagenet_unsupervised/plots/concept_image/Zoo_6673.pdf
Saved: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_cgdl_imagenet_unsupervised/plots/concep